# 03 Oxygen reduction (IA-ORR-001)

**Requirement:** `IA-ORR-001`

Patents: US12308414B2, EP4602674A1. Equations are paraphrased; see claims/paragraphs cited below.

## Governing equations


Oxygen reduction (US12308414B2):

$$\mathrm{O_2} + 2\,\mathrm{H_2O} + 4\,\mathrm{e^-} \to 4\,\mathrm{OH^-}$$

Modes: floating, vertical natural-air-breathing, inverse, tubular, stacked
submerged with depth-equalized pressure drop, wavy/rippled, bifunctional
(IA-SYS-009, IA-SYS-012; US Claims 1, 4, 7-12).
One face to electrolyte/channels, opposing face to air (EP Claim 1). PTFE/GDE parameterizable.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path
setup_path()
from plant_sim.params import default_params, CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G
from plant_sim.plotting import apply_style, timeseries, xy_plot, FIGURES
apply_style()
P = default_params()
print("6 M KOH default", P.c_KOH_mol_m3/1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G,2), round(CAPACITY_MAGNETITE_MAH_G,2))
print("figures ->", FIGURES)


## Isolated component simulation

In [ ]:
from plant_sim.components.orr import OxygenReduction, ORR_MODES, stacked_orr_flows
from plant_sim.components.base import integrate_component
print("ORR modes", ORR_MODES)
orr = OxygenReduction(P, mode="stacked_submerged")
t, y = integrate_component(lambda t,y: orr.rhs(t,y,{"I_orr_A": -20*P.I_discharge_100h_A, "T_K": P.T_ep_sim_K, "c_O2_gdl_mol_m3": 8.0, "p_O2_Pa": 21200, "a_oh": 6.0, "a_h2o": 0.72}, {}), orr.y0(), (0, 600), n_eval=120)
timeseries(t, {"c_tpb": y[1], "q_dl": y[0]}, xlabel="t (s)", ylabel="c_O2 (mol/m3) / q (C)", title="ORR TPB oxygen", callout="IA-ORR-001", stem="nb03_orr")
eq = stacked_orr_flows(P, 5e-4, True, P.T_ep_sim_K)
xy_plot(eq["z_m"], {"dP": eq["dP_Pa"]}, xlabel="z (m)", ylabel="ΔP (Pa)", title="Stacked ORR ΔP (compensated)", callout="US Claim 1", stem="nb03_dp")


## Verification against patents / SSS002

In [ ]:
print('pO2 discharge window', P.pO2_discharge_min_atm, P.pO2_discharge_max_atm, 'atm')